<a href="https://colab.research.google.com/github/Adri22K/ProjetoAndreaBD/blob/colab/Dados_API_TrilhaA_aluno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explorando uma API: dados históricos de clima

## Projeto: preparar dados para prever chuva no dia seguinte

Neste notebook, vamos aprender a:

1. entender o que é uma API e consultar sua documentação;
2. instalar e importar dependências;
3. escolher endpoint, parâmetros e variáveis;
4. fazer uma requisição HTTP com `requests`;
5. verificar status, cabeçalhos e formato da resposta;
6. interpretar um JSON e convertê-lo em `DataFrame`;
7. avaliar qualidade e realizar análises iniciais;

**Pergunta do projeto:** usando as condições meteorológicas de hoje, é possível prever se choverá amanhã?

> Este notebook realiza a coleta e a exploração inicial. O treinamento do modelo pode ser desenvolvido em uma etapa posterior.

## 1. Conceitos essenciais

- **API:** interface que permite que programas troquem dados e comandos.
- **Endpoint:** endereço que oferece um recurso específico da API.
- **Requisição:** pedido enviado pelo cliente ao servidor.
- **Resposta:** retorno do servidor, com status, cabeçalhos e corpo.
- **Parâmetros:** informações que detalham o pedido, como cidade, datas e variáveis.
- **JSON:** formato textual estruturado em pares chave–valor e listas.

Neste projeto, usaremos o endpoint:

`https://archive-api.open-meteo.com/v1/archive`

Documentação oficial: https://open-meteo.com/en/docs/historical-weather-api

A Open-Meteo informa que os dados históricos são dados de **reanálise**: observações de várias fontes são combinadas com modelos meteorológicos. Portanto, não são simplesmente medições brutas de uma única estação.

## 2. Dependências

| Biblioteca | Uso no notebook |
|---|---|
| `requests` | enviar a requisição HTTP e receber a resposta |
| `json` | exibir o JSON de maneira organizada |
| `pandas` | organizar e analisar os dados em tabela |
| `matplotlib` | construir gráficos básicos |
| `seaborn` | criar gráficos estatísticos com menos código |

No Google Colab, essas bibliotecas normalmente já estão instaladas. Em um ambiente local, execute a célula abaixo apenas se necessário.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 3. Escolher o nível de detalhe

A API permite solicitar:

- `hourly`: um registro por hora;
- `daily`: variáveis agregadas por dia.

Para o primeiro projeto, vamos usar `daily`. Assim, cada linha do futuro `DataFrame` representará um dia, sem a necessidade de agregar 24 registros horários.

### Variáveis diárias escolhidas

| Variável | Significado | Por que pode ajudar? |
|---|---|---|
| `temperature_2m_max` | temperatura máxima | caracteriza as condições térmicas |
| `temperature_2m_min` | temperatura mínima | permite observar amplitude térmica |
| `temperature_2m_mean` | temperatura média | resume a temperatura diária |
| `relative_humidity_2m_mean` | umidade relativa média | ar úmido favorece formação de nuvens e precipitação |
| `cloud_cover_mean` | cobertura média de nuvens | aproxima a condição de nebulosidade |
| `pressure_msl_mean` | pressão média ao nível do mar | mudanças de pressão podem indicar sistemas meteorológicos |
| `precipitation_sum` | precipitação total do dia | chuva pode persistir de um dia para outro |
| `precipitation_hours` | horas com precipitação | indica duração da chuva atual |
| `wind_speed_10m_max` | maior velocidade do vento | pode acompanhar frentes e mudanças do tempo |
| `shortwave_radiation_sum` | radiação solar diária | tende a diminuir em dias mais nublados |
| `sunshine_duration` | duração do brilho solar | indicador indireto de nebulosidade |

> A existência de relação física não garante que uma variável será útil ao modelo. Isso será avaliado posteriormente com dados e métricas.

In [ ]:
variaveis_diarias = [
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "cloud_cover_mean",
    "pressure_msl_mean",
    "precipitation_sum",
    "precipitation_hours",
    "wind_speed_10m_max",
    "shortwave_radiation_sum",
    "sunshine_duration",
]

variaveis_diarias

## 4. Definir os parâmetros da requisição

Usaremos as coordenadas aproximadas da cidade de São Paulo. Coordenadas são mais precisas que nomes de cidades e são exigidas por esse endpoint.

### Parâmetros obrigatórios

- `latitude` e `longitude`: localização geográfica;
- `start_date` e `end_date`: intervalo no formato `AAAA-MM-DD`;
- `daily`: variáveis que desejamos obter;
- `timezone`: necessária para variáveis diárias e importante para interpretar corretamente as datas.

O período de 2015 a 2025 gera milhares de dias: é suficiente para uma primeira experiência, mas ainda leve para um notebook didático.

In [ ]:
url = "https://archive-api.open-meteo.com/v1/archive"

parametros = {
    "latitude": -23.55,
    "longitude": -46.63,
    "start_date": "2015-01-01",
    "end_date": "2025-12-31",
    "daily": variaveis_diarias,  #parametros definidos anteriormente
    "timezone": "America/Sao_Paulo",
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm",
}

parametros

### Atividade rápida

Antes de executar a requisição, responda:

1. Qual é o endpoint?

```
Endpoint é o endereço usado pelo sistema para acessar e solicitar dados de uma API.
```
2. Quais parâmetros identificam o lugar?
```
Os parâmetros que identificam o lugar são:

latitude: posição geográfica norte/sul;
longitude: posição geográfica leste/oeste.
```
3. Quais parâmetros definem o período?
```
Os parâmetros que definem o período são:

start_date: data inicial;
end_date: data final;
forecast_days: quantidade de dias da previsão;
past_days: quantidade de dias anteriores.
```
4. Por que foi escolhido `daily`, e não `hourly`?
```
Foi escolhido daily porque o projeto analisa os dados por dia, facilitando a integração com as informações meteorológicas da NASA POWER e da Open-Meteo Historical. Como a API de qualidade do ar fornece dados em hourly, é necessário calcular a média diária dos poluentes usando o Pandas.
```
5. Quantos dias você espera receber aproximadamente?
```
7 dias.
```

## 5. Requisição HTTP

`requests.get()` envia uma requisição do tipo **GET**, usada para consultar um recurso.

- `params=parametros` monta a *query string* corretamente;
- `timeout=30` impede que o programa espere indefinidamente;
- o resultado é um objeto `Response`, ainda não um dicionário Python.

In [ ]:
try:
    resposta = requests.get(
        url,
        params=parametros,
        timeout=30)
except requests.exceptions.RequestException as erro:
    raise RuntimeError(f"Não foi possível acessar a API: {erro}") from erro

print("Tipo do objeto:", type(resposta))
print("URL montada:", resposta.url)
print("Status HTTP:", resposta.status_code)
print("Content-Type:", resposta.headers.get("Content-Type"))

## 6. Interpretar a resposta

Alguns status HTTP frequentes:

| Status | Significado |
|---:|---|
| 200 | requisição concluída com sucesso |
| 400 | parâmetros ausentes ou incorretos |
| 404 | recurso não encontrado |
| 429 | limite de requisições excedido |
| 500 | erro interno do servidor |

`raise_for_status()` interrompe a execução se o status indicar erro. Antes de chamar `.json()`, também verificaremos se o servidor declarou conteúdo JSON.

In [ ]:
resposta.raise_for_status()

content_type = resposta.headers.get("Content-Type", "").lower()
if "json" not in content_type:
    raise ValueError(
        "A API não retornou JSON. "
        f"Content-Type recebido: {content_type or 'não informado'}"
    )

dados = resposta.json()

print("Tipo de resposta:", type(resposta))
print("Tipo após .json():", type(dados))
print("Chaves principais:", list(dados.keys()))

In [ ]:
# Visualização parcial e organizada do documento JSON.
# Limitei a saída para não ocupar muitas páginas do notebook.
texto_json = json.dumps(dados, indent=2, ensure_ascii=False)
print(texto_json[:1000])

## 7. Entender a estrutura recebida

O JSON tem informações gerais no primeiro nível. Os dados tabulares estão em `daily`, enquanto `daily_units` informa as unidades.

Observe uma característica importante: dentro de `daily`, cada chave contém uma lista. Os elementos na mesma posição pertencem ao mesmo dia. É essa estrutura que o Pandas converterá em linhas e colunas.

In [ ]:
print("Metadados:")
print("Coordenada retornada:", dados["latitude"], dados["longitude"])
print("Fuso horário:", dados["timezone"])
print("Elevação:", dados["elevation"], "m")

print("\nUnidades das variáveis:")
display(pd.Series(dados["daily_units"], name="unidade").to_frame())

## 8. Converter o JSON para DataFrame

Um `DataFrame` organiza os dados em linhas e colunas e facilita filtros, cálculos, gráficos e preparação para aprendizado de máquina.

In [ ]:
df = pd.DataFrame(dados["daily"])
df["time"] = pd.to_datetime(df["time"])

df.head()

In [ ]:
print(dados.keys())

In [ ]:
print("Dimensão (linhas, colunas):", df.shape)
print("Período:", df["time"].min().date(), "até", df["time"].max().date())
print("\nTipos de dados:")
display(df.dtypes.to_frame("tipo"))

## 9. Auditoria inicial da qualidade

---



Antes de analisar ou treinar um modelo, devemos verificar:

- datas repetidas;
- datas ausentes no intervalo;
- valores ausentes (`NaN`);
- linhas duplicadas
- verificar a dimensão do DataFrame
- verificar as primeiras linhas
- verificar as ultimas linhas
- verificar nome das colunas
- usar info()
- usar describe()
- visualizar os dados de 1 coluna
- visualuzar os dados de 2 colunas
- usar o loc e o iloc
- Quantos dias a precipitação foi 0
